# 03 — Quartiers : contours et rattachement

**Étape du pipeline** : *Raw data → Data transformation*, pour la dimension spatiale.

Le but est de donner aux deux sources un **même référentiel de quartiers**, pour qu'une carte choroplèthe puisse les comparer zone par zone.

| Ville | Niveau retenu | Source | Licence |
|---|---|---|---|
| Paris | 20 arrondissements | Ville de Paris, opendata.paris.fr | ODbL |
| Berlin | 12 Bezirke | Geoportal Berlin (ALKIS) | DL-DE Zero 2.0 |
| Londres | 33 boroughs | London Datastore, Statistical GIS Boundary Files | OGL (contient des données OS) |
| Rome | 15 municipi | OpenStreetMap (Overpass) | ODbL |

Pourquoi les municipi à Rome (D16) : les noms Booking de Rome sont des repères touristiques (« Spagna », « Trevi », « Central Station »…) qui mélangent de petits rioni et de grands quartieri. Les municipi sont des zones de taille comparable, et chaque nom Booking s'y rattache.

Sorties :
- `data/external/quartiers_<ville>.geojson` : contours normalisés, en pleine précision, utilisés pour la jointure spatiale (notebook 05) ;
- `data/external/correspondance_quartiers.csv` : table écrite à la main qui associe chaque nom Booking à un contour ;
- `data/processed/quartiers_<ville>.geojson` : contours simplifiés et orientés pour `d3.geoPath`.

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prep").is_dir())
sys.path.insert(0, str(RACINE))
RAW, INTERIM, PROCESSED = (RACINE / "data" / d for d in ("raw", "interim", "processed"))
EXTERNAL = RACINE / "data" / "external"

import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from src.prep import quartiers, schema
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## 1. Téléchargement des contours bruts

In [ ]:
if not (RAW / "geo" / "rome_municipi_osm.json").exists():
    subprocess.run([str(RACINE / "scripts" / "download_geo.sh")], check=True)
sorted(p.name for p in (RAW / "geo").iterdir())

## 2. Contours normalisés

Chaque ville est ramenée au même format : `ville`, `quartier` (libellé affiché), `code` (code officiel), `ordre`, en WGS84.

In [ ]:
contours = quartiers.charger_contours(RAW / "geo")
print("géométries valides :", contours.geometry.is_valid.all())
contours.groupby("ville").agg(zones=("quartier", "size"), exemple=("quartier", "first"))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (v, g) in zip(axes, contours.groupby("ville")):
    g.plot(ax=ax, color="#dde5ee", edgecolor="#3d5a80", linewidth=.6)
    ax.set_title(v); ax.set_axis_off()
plt.tight_layout()

## 3. Correspondance des noms Booking

Les hôtels n'ont pas encore de coordonnées : en attendant le géocodage (notebook 04), leur quartier vient du nom Booking.

In [ ]:
correspondance = quartiers.charger_correspondance(EXTERNAL / "correspondance_quartiers.csv")
print("noms sans contour :", set(correspondance["quartier"]) - set(contours["quartier"]) or "aucun")
correspondance.astype({"nb_hotels": int}).pivot_table(index="ville", columns="methode", values="nb_hotels", aggfunc="sum", fill_value=0)

In [ ]:
correspondance[correspondance["methode"] != "identique"]

- **identique** : le nom Booking est celui du contour.
- **contenance** : un quartier non administratif rattaché au contour qui le contient, par exemple Chiswick → Hounslow ou Trevi → Municipio I.
- **approximation** : un nom à cheval sur plusieurs contours, ou trop vague, par exemple « Rome City Center » → Municipio I, ou « Tiburtino », qui s'étend sur trois municipi.

Les approximations concernent 36 hôtels à Rome et 1 à Londres. Elles disparaîtront avec le géocodage.

## 4. Rattachement des Airbnb par jointure spatiale

In [ ]:
airbnb = schema.conformer(pd.read_parquet(INTERIM / "airbnb.parquet"))
r = quartiers.rattacher_points(airbnb, contours)
pd.crosstab(airbnb["ville"], r["methode_geo"].fillna("non rattaché"), margins=True)

- **contenu** : le point est dans le contour.
- **proche** : le point est hors de tout contour, mais à moins de 250 m de l'un d'eux. Il s'agit surtout d'annonces au bord de la Tamise, que les boroughs de Londres excluent (« Excluding MHW »).

## 5. Couverture : où peut-on comparer ?

In [ ]:
hotels = schema.conformer(pd.read_parquet(INTERIM / "hotels.parquet"))
q_hotels = quartiers.rattacher_noms(hotels, correspondance)
couv = pd.concat({
    "Airbnb": r["quartier_geo"].groupby([airbnb["ville"], r["quartier_geo"]], observed=True).size(),
    "Hôtel": q_hotels.groupby([hotels["ville"], q_hotels], observed=True).size(),
}, axis=1).reindex(pd.MultiIndex.from_frame(contours[["ville", "quartier"]])).fillna(0).astype(int)
couv["comparable"] = (couv["Airbnb"] >= 30) & (couv["Hôtel"] >= 10)
couv.groupby(level="ville").agg(zones=("Airbnb", "size"), avec_airbnb=("Airbnb", lambda s: (s > 0).sum()),
                                 avec_hotels=("Hôtel", lambda s: (s > 0).sum()), comparables=("comparable", "sum"))

In [ ]:
couv.loc["Rome"].T

**Constats**

- **Paris** (19 zones comparables sur 20) et **Berlin** (10 sur 12) se prêtent bien à une choroplèthe.
- **Londres** : le dataset Airbnb ne couvre que 14 boroughs centraux sur 33, et 10 sont comparables. Les boroughs périphériques sont vides côté Airbnb, il faudra les griser.
- **Rome** : 156 hôtels sur 207 sont dans le Municipio I, et seul celui-ci est comparable. La carte de Rome reste utile pour les Airbnb, mais pas pour la comparaison entre les deux sources par zone. C'est une limite à signaler, qui s'ajoute à celle des prix (D7).

## 6. Exports

In [ ]:
ext = quartiers.exporter_contours(contours, EXTERNAL, decimales=6)
d3 = quartiers.exporter_contours(quartiers.pour_d3(contours), PROCESSED, decimales=5)
for p in ext + d3:
    print(f"{p.relative_to(RACINE)}  {p.stat().st_size / 1e3:.0f} Ko")

Côté D3, un fichier se charge et se dessine ainsi :

```js
const zones = await d3.json("data/processed/quartiers_paris.geojson");
const projection = d3.geoMercator().fitSize([largeur, hauteur], zones);
svg.selectAll("path").data(zones.features).join("path").attr("d", d3.geoPath(projection));
```

Les anneaux extérieurs sont orientés dans le sens horaire, comme l'attend `d3-geo`. Avec l'orientation standard de GeoJSON, D3 dessinerait le globe entier moins le quartier.